# Model Card — Modèle de maintenance prédictive (format Hugging Face)

Ce notebook remplit automatiquement le template de model card Hugging Face (`templates/modelcard_template.md`) à partir du **modèle validé** dans les explorations précédentes : XGBoost, horizon 24h, meilleurs hyperparamètres issus de la recherche Optuna, seuil calibré, mesures CodeCarbon, et explicabilité SHAP.

Rédigé pour un lecteur **non technique** : chaque section du template est remplie en langage clair, sans jargon non expliqué.

**Optimisation du temps de traitement** : aucune nouvelle recherche d'hyperparamètres ici — réutilisation directe des meilleurs réglages déjà trouvés (`BEST_PARAMS`), un seul entraînement de calibration + un seul entraînement final, un échantillon réduit pour SHAP. Pas de redécouverte de ce qui est déjà su.

In [21]:
import time
_notebook_start_time = time.time()

import numpy as np
import pandas as pd
import shap
from sklearn.metrics import average_precision_score, roc_auc_score

from indusense.common.config import RANDOM_SEED, OUTPUT_DIR, load_config
from indusense.common.emissions import make_tracker, get_last_energy_kwh
from indusense.common.modelcard import make_model_id, render_model_card
from indusense.ml.data import load_gold_dataset, temporal_split
from indusense.ml.train import build_tuned_model, train_final_model, save_model

SEED = RANDOM_SEED

import os
N_CORES = os.cpu_count() or 1  # utilisé dans les sections Hardware/Compute Infrastructure de la model card


## 1. Charger les données et le modèle validé

Mêmes exclusions de colonnes, même horizon (24h) que les notebooks précédents. `BEST_PARAMS` reprend exactement les hyperparamètres retenus à l'issue de la recherche Optuna — pas une nouvelle recherche.

In [22]:
df, feats, horizon = load_gold_dataset()
(X_train, y_train), (X_val, y_val), (X_test, y_test) = temporal_split(df, feats, horizon)

cfg_train = load_config("train", domain="ml")
BEST_PARAMS = cfg_train["hyperparameters"]  # meilleurs hyperparamètres retenus (recherche Optuna précédente)

print(f"{len(feats)} features — train: {len(X_train)}, val: {len(X_val)}, test: {len(X_test)}")


71 features — train: 93990, val: 20145, test: 20145


## 2. Identifiant unique de modèle (ID + version)

Chaque combinaison de réglages produit un identifiant stable et reproductible : un hash court calculé à partir des hyperparamètres eux-mêmes (pas d'un tirage aléatoire) — deux exécutions avec les mêmes réglages donnent le même identifiant, ce qui permet de tracer précisément *quelle* version du modèle est documentée par cette model card.

In [23]:
import datetime

MODEL_VERSION = load_config("modelcard", domain="common")["model_version"]
MODEL_ID = make_model_id("predictive-maintenance-xgboost-24h", BEST_PARAMS)
CARD_DATE = datetime.date.today().isoformat()

print(f"MODEL_ID   : {MODEL_ID}")
print(f"Date de la fiche : {CARD_DATE}")


MODEL_ID   : predictive-maintenance-xgboost-24h-v1.0.0-5f2e2b2a
Date de la fiche : 2026-07-23


## 3. Entraînement du modèle final (instrumenté CodeCarbon) + calibration du seuil

Deux modèles, pour rester rigoureux :
- **Modèle de calibration** : entraîné sur `train` seul, sert uniquement à calibrer le seuil de décision sur `validation` (jamais entraîné sur les données qu'il calibre — pas de fuite).
- **Modèle final** : entraîné sur `train + validation` réunis (toutes les données disponibles avant le test), c'est celui documenté par cette model card et celui dont l'entraînement est mesuré par CodeCarbon.

In [24]:
# --- Modèle de calibration (train seul) : seuil calibré sur validation, jamais entraîné dessus ---
calib_model = build_tuned_model()
calib_model.fit(X_train, y_train)
from indusense.ml.evaluate import calibrate_threshold, evaluate_at_threshold

threshold = calibrate_threshold(calib_model, X_val, y_val)
THRESHOLD_PERCENTILE = load_config("evaluate", domain="ml")["threshold_percentile"]

# --- Modèle final (train + validation), instrumenté CodeCarbon ---
tracker = make_tracker("model_card_final_training")
tracker.start()
_t0 = time.time()

final_model = train_final_model(X_train, y_train, X_val, y_val)

training_duration_s = time.time() - _t0
emissions_kg = tracker.stop()
training_kwh = get_last_energy_kwh("model_card_final_training")

print(f"Entraînement final : {training_duration_s:.2f}s  —  {training_kwh:.6f} kWh  —  {emissions_kg*1000:.4f} gCO2eq")


Entraînement final : 7.26s  —  0.000075 kWh  —  0.0042 gCO2eq


## 4. Métriques de test (au seuil calibré)

In [25]:
test_metrics = evaluate_at_threshold(y_test, threshold, model=final_model, X=X_test)
prauc_test, auc_test = test_metrics["prauc"], test_metrics["auroc"]
recall_test, precision_test, fpr_test = test_metrics["recall"], test_metrics["precision"], test_metrics["fpr"]

print(f"PR-AUC (test)        : {prauc_test:.4f}")
print(f"AUC-ROC (test)       : {auc_test:.4f}")
print(f"Seuil de décision    : {threshold:.4f}  (percentile {THRESHOLD_PERCENTILE} des scores sur les machines saines de validation)")
print(f"Rappel (test)        : {recall_test:.2%}  — proportion de vraies pannes détectées")
print(f"Précision (test)     : {precision_test:.2%}  — proportion d'alertes réellement suivies d'une panne")
print(f"Faux positifs (test) : {fpr_test:.2%}  — proportion de machines saines déclenchant une fausse alerte")


PR-AUC (test)        : 0.4488
AUC-ROC (test)       : 0.6811
Seuil de décision    : 0.4042  (percentile 95 des scores sur les machines saines de validation)
Rappel (test)        : 28.98%  — proportion de vraies pannes détectées
Précision (test)     : 55.02%  — proportion d'alertes réellement suivies d'une panne
Faux positifs (test) : 4.94%  — proportion de machines saines déclenchant une fausse alerte


## 5. Explicabilité (SHAP) — pour la section « Model Examination »

Reprend la démarche du notebook `shap_explainability_ml.ipynb` — échantillon réduit pour rester rapide, uniquement pour synthétiser les features dominantes et le niveau de concentration de l'impact, à inclure dans la model card.

In [26]:
from indusense.ml.explain import compute_shap_values, concentration_share

shap_values, X_shap_sample = compute_shap_values(final_model, X_val)

mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
top5_idx = np.argsort(mean_abs_shap)[-5:][::-1]
top5_features = [feats[i] for i in top5_idx]

top10_share = concentration_share(shap_values, top_n=10)

print("Top 5 features (impact SHAP moyen) :", top5_features)
print(f"Part de l'impact total couverte par les 10 premières features : {top10_share:.1%}")


Top 5 features (impact SHAP moyen) : ['temp_mean_24h', 'pressure_mean_24h', 'voltage_mean_24h', 'rotation_max_24h', 'temp_max_24h']
Part de l'impact total couverte par les 10 premières features : 57.4%


## 6. Construction du contenu de la model card

Chaque section est rédigée en langage clair, pour un lecteur non technique — les chiffres exacts (métriques, seuil, émissions) sont insérés automatiquement à partir des calculs ci-dessus, jamais recopiés à la main.

In [27]:
card_context = {}

# --- Métadonnées YAML (en-tête du template) ---
card_context["card_data"] = f"""language: fr
license: proprietary
tags:
  - tabular-classification
  - predictive-maintenance
  - xgboost
model_id: {MODEL_ID}
model_version: {MODEL_VERSION}"""

card_context["model_id"] = MODEL_ID

card_context["model_summary"] = (
    "Ce modèle prédit si une machine industrielle risque de tomber en panne dans les **24 prochaines heures**, "
    "à partir de mesures de capteurs (température, tension électrique, vitesse de rotation, pression, volume produit). "
    "Il est destiné à déclencher des alertes de maintenance préventive, pas à remplacer une inspection humaine."
)

In [28]:
card_context["model_description"] = (
    f"Modèle de classification binaire (XGBoost) entraîné pour prédire la probabilité qu'une machine tombe en panne "
    f"dans les 24 heures suivant une mesure, à partir de {len(feats)} variables décrivant l'état récent de la machine "
    f"(températures, tensions, vibrations, pression, volume de production, historique de maintenance à moyen terme). "
    f"Le modèle a été sélectionné après une recherche raisonnée d'hyperparamètres (Optuna, voir section Training Procedure), "
    f"puis documenté avec une analyse d'explicabilité (SHAP) et une mesure de son coût énergétique d'entraînement (CodeCarbon)."
)

card_context["developers"] = "Équipe data / IA — projet de maintenance prédictive (voir Model Card Authors)"
card_context["funded_by"] = "[More Information Needed]"
card_context["shared_by"] = "[More Information Needed]"
card_context["model_type"] = "Classification binaire supervisée sur données tabulaires (gradient boosting, XGBoost)"
card_context["language"] = "Non applicable — modèle sur données numériques de capteurs, pas de traitement de langage naturel"
card_context["license"] = "À définir selon la politique de l'organisation (proprietary par défaut)"
card_context["base_model"] = "Aucun — entraîné from scratch, pas un modèle pré-entraîné affiné"

card_context["repo"] = "training-4-aelion (branche develop)"
card_context["paper"] = "[More Information Needed]"
card_context["demo"] = "[More Information Needed]"

In [29]:
# --- Uses : Direct Use et Out-of-Scope Use (priorité demandée) ---
card_context["direct_use"] = (
    "Ce modèle est destiné à être utilisé **tel quel** pour générer un score de risque de panne à 24h par machine et par "
    "heure, à partir des mêmes variables que celles utilisées à l'entraînement (mêmes capteurs, même fréquence de mesure). "
    "Usage prévu : alimenter un tableau de bord de maintenance préventive, ou déclencher une alerte à destination d'un "
    "technicien lorsque le score dépasse le seuil calibré (voir section Evaluation). **Le modèle ne doit jamais prendre "
    "seul une décision d'arrêt de machine** — il sert à prioriser l'attention humaine, pas à s'y substituer."
)

card_context["downstream_use"] = (
    "Ré-entraînement possible sur un parc de machines différent, à condition de vérifier que la distribution des capteurs "
    "et le taux de pannes restent comparables au dataset d'origine (voir Bias, Risks, and Limitations). "
    "Un ajustement du seuil de décision (actuellement calibré au 95e percentile des scores sur machines saines) est "
    "recommandé si le contexte opérationnel change (coût d'une panne manquée vs coût d'une fausse alerte)."
)

card_context["out_of_scope_use"] = (
    "**Ce modèle ne doit pas être utilisé** :\n\n"
    "- Pour des machines d'un type différent de celles du parc d'entraînement (autres capteurs, autre régime de fonctionnement), "
    "sans revalidation complète — aucune garantie de généralisation à un contexte industriel différent.\n"
    "- Pour prédire un horizon différent de 24h sans ré-entraînement spécifique (un modèle entraîné sur l'horizon 24h "
    "n'est pas fiable pour prédire à 6h ou 48h — voir les notebooks d'exploration des autres horizons).\n"
    "- Comme seule source de décision pour arrêter une machine, engager une intervention coûteuse, ou toute décision "
    "ayant un impact sur la sécurité des personnes — le rappel du modèle est **partiel** (voir Limitations), une partie "
    "des pannes réelles ne sera pas détectée.\n"
    "- En dehors du contexte de maintenance industrielle pour lequel il a été conçu (aucune vocation à être réutilisé "
    "sur un autre type de problème de classification, même à structure de données similaire, sans revalidation)."
)

In [30]:
# --- Bias, Risks, and Limitations : honnête, chiffré, pas édulcoré ---
card_context["bias_risks_limitations"] = f"""
**Signal faible et diffus.** L'analyse d'explicabilité (SHAP) montre qu'aucune variable ne domine la décision du modèle : les 5 variables les plus influentes n'expliquent qu'une partie modérée de l'impact total, et il faut regrouper une dizaine de variables pour couvrir environ {top10_share:.0%} de l'impact ({', '.join(top5_features[:3])}, entre autres). Autrement dit, le modèle combine de nombreux indices faibles plutôt que de s'appuyer sur un signal fort et unique — un profil cohérent avec la nature progressive d'une dégradation mécanique, mais qui limite la performance atteignable.

**Faux négatifs (pannes manquées).** Au seuil de décision retenu, le modèle détecte environ {recall_test:.0%} des vraies pannes sur le jeu de test — ce qui signifie qu'environ {1-recall_test:.0%} des pannes réelles à 24h **ne sont pas détectées** par le modèle. C'est la limitation la plus importante à communiquer à tout utilisateur : ce modèle réduit le risque, il ne l'élimine pas.

**Faux positifs (fausses alertes).** À l'inverse, environ {fpr_test:.0%} des machines saines déclenchent une alerte à tort à ce même seuil — un coût opérationnel (temps d'inspection sans panne trouvée) à mettre en balance avec le coût d'une panne manquée.

**Instabilité observée lors des tests de robustesse.** Des essais antérieurs sur ce même type de modèle ont montré que les résultats (quel algorithme domine, quelle importance de variable ressort) peuvent varier sensiblement selon la taille du split de validation et l'initialisation aléatoire — un signal de prudence sur la robustesse du classement exact des variables, même si le modèle retenu ici reste cohérent sur ses métriques globales.

**Portée des données d'entraînement.** Le modèle n'a vu que les machines, la période et les conditions opérationnelles présentes dans `gold_dataset`. Aucune garantie de performance sur un autre parc de machines, une autre période de l'année, ou des conditions de fonctionnement inhabituelles (arrêts prolongés, changements de process) non représentées dans les données d'entraînement.
"""

card_context["bias_recommendations"] = (
    "Utiliser ce modèle comme **aide à la priorisation**, jamais comme décision automatique unique. Toute alerte doit "
    "être confirmée par une inspection humaine avant action. Le seuil de décision doit être révisé périodiquement "
    "(nouvelles données, changement de coût métier entre panne manquée et fausse alerte). Un ré-entraînement périodique "
    "est recommandé pour éviter une dérive du modèle si les conditions opérationnelles évoluent. Ne pas extrapoler la "
    "performance mesurée ici à un parc de machines ou un horizon de prédiction différents sans revalidation."
)

In [31]:
card_context["get_started_code"] = f"""```python
import xgboost as xgb

model = xgb.XGBClassifier()
model.load_model("{MODEL_ID}.json")

# X : DataFrame avec les mêmes {len(feats)} colonnes que celles utilisées à l'entraînement
proba = model.predict_proba(X)[:, 1]
alerte = proba > {threshold:.4f}  # seuil calibré, voir section Evaluation
```"""

In [32]:
# --- Training Details ---
card_context["training_data"] = (
    f"`gold_dataset` — mesures capteurs agrégées par machine et par heure (température, tension, vitesse de rotation, "
    f"pression, volume produit, taux d'utilisation), avec un historique de maintenance à moyen terme. "
    f"{len(X_train)} lignes d'entraînement, {len(X_val)} de validation, {len(X_test)} de test, découpées dans le temps "
    f"(le test correspond toujours à la période la plus récente, jamais mélangée avec l'entraînement). "
    f"{len(feats)} variables conservées après exclusion des colonnes qui donneraient au modèle une information sur le "
    f"futur (fuite de données) — voir la checklist anti-fuite du projet."
)

card_context["preprocessing"] = (
    "Exclusion systématique des identifiants, des colonnes dérivées du futur par construction, et des labels des autres "
    "horizons de prédiction (6h, 12h, 48h). Pas de normalisation des variables (XGBoost n'en a pas besoin, à la différence "
    "des réseaux de neurones). Rééquilibrage du déséquilibre de classe géré par le poids `scale_pos_weight`, pas par "
    "sur/sous-échantillonnage des données."
)

card_context["training_regime"] = "CPU, précision flottante standard (fp32) — pas d'entraînement sur GPU, pas de calcul en précision réduite."

card_context["speeds_sizes_times"] = (
    f"Entraînement du modèle final : {training_duration_s:.2f} secondes sur un seul cœur de processeur "
    f"({training_kwh:.6f} kWh consommés). Les hyperparamètres eux-mêmes ont été obtenus via une recherche Optuna "
    f"antérieure (20 essais, ~3 minutes) — non refaite ici, réutilisée telle quelle (voir "
    f"`optuna_xgboost_predictive_maintenance.ipynb`)."
)

In [33]:
# --- Evaluation ---
card_context["testing_data"] = (
    f"Sous-ensemble `test` de `gold_dataset` ({len(X_test)} lignes), correspondant à la période la plus récente, "
    f"jamais utilisée ni pendant l'entraînement ni pendant la calibration du seuil de décision."
)

card_context["testing_factors"] = (
    "Évaluation globale sur l'ensemble du jeu de test — pas de décomposition par machine individuelle ou par sous-période "
    "dans cette version de la model card. Un futur travail pourrait vérifier la stabilité de la performance par machine."
)

card_context["testing_metrics"] = (
    "**PR-AUC** (aire sous la courbe précision-rappel) — métrique principale, adaptée au déséquilibre de classe "
    "(pannes rares). **AUC-ROC** — métrique complémentaire, moins sensible au déséquilibre. **Rappel, précision et "
    "taux de faux positifs** au seuil de décision retenu — pour une lecture opérationnelle directe (combien de pannes "
    "détectées, combien de fausses alertes)."
)

card_context["results"] = f"""
| Métrique | Valeur (jeu de test) |
|---|---|
| PR-AUC | {prauc_test:.4f} |
| AUC-ROC | {auc_test:.4f} |
| Seuil de décision | {threshold:.4f} (percentile {THRESHOLD_PERCENTILE} des scores sur machines saines de validation) |
| Rappel (pannes détectées) | {recall_test:.1%} |
| Précision (alertes confirmées) | {precision_test:.1%} |
| Faux positifs (fausses alertes) | {fpr_test:.1%} |
"""

card_context["results_summary"] = (
    f"Le modèle détecte environ {recall_test:.0%} des pannes réelles à 24h, avec un taux de fausses alertes d'environ "
    f"{fpr_test:.0%} sur les machines saines, au seuil retenu. La performance est modeste mais honnête au regard du "
    f"signal disponible (voir Model Examination) — un compromis assumé plutôt qu'une sur-promesse."
)

In [34]:
# --- Model Examination (SHAP) ---
card_context["model_examination"] = f"""
Une analyse d'explicabilité (SHAP, voir `shap_explainability_ml.ipynb`) a été menée sur ce modèle :

- **Variables dominantes** : {', '.join(top5_features)} — des agrégats sur 24h de température, tension et rotation, cohérents avec une dégradation mécanique progressive plutôt qu'un pic isolé.
- **Concentration de l'impact** : diffus plutôt que concentré — les 10 premières variables couvrent environ {top10_share:.0%} de l'impact total sur les prédictions, aucune variable ne domine à elle seule.
- **Vérification anti-fuite** : aucune des variables dominantes ne présente de corrélation suspecte avec les colonnes explicitement exclues pour fuite de données — les variables qui pilotent la décision sont mécaniquement plausibles, pas des raccourcis statistiques douteux.
- **Cohérence métier** : les variables dominantes correspondent à des mesures capteur reconnaissables (température, tension, rotation, pression, volume produit), pas à des artefacts de construction du dataset.
"""

In [35]:
# --- Environmental Impact (CodeCarbon) ---
card_context["hardware_type"] = f"CPU — {N_CORES} cœur(s) détecté(s) et utilisé(s) pour l'entraînement (pas de GPU)"
card_context["hours_used"] = f"{training_duration_s/3600:.5f} heures ({training_duration_s:.2f} secondes) pour l'entraînement du modèle final"
card_context["cloud_provider"] = "[More Information Needed]"
card_context["cloud_region"] = "France (intensité carbone du calcul forcée sur le mix électrique français, voir notebook éco-conception)"
card_context["co2_emitted"] = f"{emissions_kg*1000:.4f} grammes de CO2 équivalent (mesuré avec CodeCarbon, {training_kwh:.6f} kWh consommés)"

In [36]:
# --- Technical Specifications ---
card_context["model_specs"] = (
    f"XGBoost (gradient boosting sur arbres de décision) — "
    f"n_estimators={BEST_PARAMS['n_estimators']}, max_depth={BEST_PARAMS['max_depth']}, "
    f"learning_rate={BEST_PARAMS['learning_rate']:.4f}. Objectif : classification binaire (panne / pas de panne à 24h), "
    f"loss = log-vraisemblance binaire pondérée (`scale_pos_weight={BEST_PARAMS['scale_pos_weight']:.3f}` pour compenser "
    f"le déséquilibre de classe)."
)

card_context["compute_infrastructure"] = (
    f"Poste de calcul standard (CPU, {N_CORES} cœur(s) disponibles et utilisés), pas d'infrastructure distribuée "
    f"ni de GPU nécessaire pour ce type de modèle."
)
card_context["hardware_requirements"] = "Minimal — inférence quasi instantanée sur CPU, pas de carte graphique requise, empreinte mémoire réduite (modèle < 1 Mo)."
card_context["software"] = "Python, XGBoost, scikit-learn, Optuna (recherche d'hyperparamètres), SHAP (explicabilité), CodeCarbon (mesure d'impact), MLflow (suivi des expériences)."

card_context["citation_bibtex"] = f"""```bibtex
@misc{{{MODEL_ID.replace('-', '_')},
  title = {{Predictive Maintenance XGBoost Model (24h horizon)}},
  author = {{[More Information Needed]}},
  year = {{{CARD_DATE[:4]}}},
  howpublished = {{Internal model card}},
  note = {{Model ID: {MODEL_ID}}}
}}
```"""
card_context["citation_apa"] = f"[More Information Needed] ({CARD_DATE[:4]}). Predictive Maintenance XGBoost Model (24h horizon), {MODEL_ID}."

In [37]:
# --- Glossary ---
card_context["glossary"] = """
- **PR-AUC** : aire sous la courbe précision-rappel — mesure la capacité à repérer les cas rares (pannes) sans trop de fausses alertes.
- **AUC-ROC** : aire sous la courbe ROC — mesure classique de séparation entre classes, moins sensible au déséquilibre.
- **Rappel** : proportion de vraies pannes effectivement détectées par le modèle.
- **Précision** : proportion d'alertes du modèle réellement suivies d'une panne.
- **Faux positif** : une alerte déclenchée sur une machine qui, en réalité, ne tombera pas en panne.
- **Faux négatif** : une panne réelle que le modèle n'a pas détectée.
- **SHAP** : méthode d'explicabilité qui attribue à chaque variable sa contribution à une prédiction donnée.
- **Fuite de données (data leakage)** : quand une variable d'entraînement contient, par erreur, une information sur le futur que le modèle ne devrait pas connaître au moment de la prédiction.
- **kWh / gCO2eq** : unités de consommation électrique et d'impact climatique équivalent.
"""

card_context["more_information"] = (
    "Voir les notebooks associés du projet : `optuna_xgboost_predictive_maintenance.ipynb` (recherche d'hyperparamètres), "
    "`eco_conception_codecarbon.ipynb` (mesure d'impact comparée ML/DL), `shap_explainability_ml.ipynb` (explicabilité détaillée)."
)

card_context["model_card_authors"] = "[More Information Needed]"
card_context["model_card_contact"] = "[More Information Needed]"

## 7. Rendu de la model card (Jinja2)

Le template Hugging Face (`templates/modelcard_template.md`) utilise la syntaxe Jinja2 standard (`{{ variable | default(...) }}`) — on le rend directement avec le contexte construit ci-dessus, sans réécrire le template à la main.

In [38]:
output_path = render_model_card(card_context, MODEL_ID)

print(f"Model card générée : {output_path}")
print(f"Taille : {output_path.stat().st_size} octets")


Model card générée : artifacts/ingestions/output\model_card_predictive-maintenance-xgboost-24h-v1.0.0-5f2e2b2a.md
Taille : 17093 caractères


## 8. Sauvegarde du modèle et traçabilité MLflow

In [39]:
import mlflow

model_path = save_model(final_model, MODEL_ID)  # sauvegarde du booster natif — cf. indusense.ml.train.save_model

mlflow.set_tracking_uri("sqlite:///mlflow/mlflow.db")
mlflow.set_experiment("xgboost_optuna_predictive_maintenance")

with mlflow.start_run(run_name=f"model_card_{MODEL_ID}"):
    mlflow.log_params({**BEST_PARAMS, "model_id": MODEL_ID, "horizon": horizon})
    mlflow.log_metrics({
        "prauc_test": prauc_test, "auc_test": auc_test, "recall_test": recall_test,
        "fpr_test": fpr_test, "precision_test": precision_test,
        "training_kwh": training_kwh, "training_gco2eq": emissions_kg * 1000,
        "shap_top10_share": top10_share,
    })
    mlflow.log_artifact(str(output_path))
    mlflow.log_artifact(str(model_path))

print(f"Modèle sauvegardé : {model_path}")
print(f"Run MLflow loggé : model_card_{MODEL_ID}")


Modèle sauvegardé : artifacts/ingestions/output\predictive-maintenance-xgboost-24h-v1.0.0-5f2e2b2a.json
Run MLflow loggé : model_card_predictive-maintenance-xgboost-24h-v1.0.0-5f2e2b2a


## Temps de traitement global

In [40]:
_total_elapsed = time.time() - _notebook_start_time
print(f"Temps total d'exécution du notebook : {_total_elapsed:.1f} s  ({_total_elapsed/60:.1f} min)")
print(f"  dont entraînement du modèle final (mesuré CodeCarbon) : {training_duration_s:.2f} s")
print(f"\nFichiers produits dans {OUTPUT_DIR}/ :")
print(f"  - model_card_{MODEL_ID}.md  (la model card complète)")
print(f"  - {MODEL_ID}.json  (le modèle entraîné, réutilisable directement)")
print("  - emissions.csv  (détail CodeCarbon)")

Temps total d'exécution du notebook : 18.9 s  (0.3 min)
  dont entraînement du modèle final (mesuré CodeCarbon) : 7.26 s

Fichiers produits dans artifacts/ingestions/output/ :
  - model_card_predictive-maintenance-xgboost-24h-v1.0.0-5f2e2b2a.md  (la model card complète)
  - predictive-maintenance-xgboost-24h-v1.0.0-5f2e2b2a.json  (le modèle entraîné, réutilisable directement)
  - emissions.csv  (détail CodeCarbon)
